<b><font size="6" color="#E8800A">Week 7b · Bayesian regression</font></b><br>
<b><font size="4">Bayes' theorem, pointed at a number, and what a posterior buys</font></b><br>

Bayesian regression applies Bayes' theorem to a **numeric** target, here
the resale price of a car. A least-squares model returns one predicted price per
car. A Bayesian model returns, for each car, a predicted price **and a range around
it**: how far the true price is expected to fall from the prediction.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>From a point to a distribution</font>](#distribution)
* [<font color='#E8800A'>Fitting it</font>](#fit)
* [<font color='#E8800A'>What the model learned</font>](#learned)
* [<font color='#E8800A'>The prior is a ridge penalty</font>](#prior-is-penalty)
* [<font color='#E8800A'>One car's prediction and its range</font>](#one-car)
* [<font color='#E8800A'>What the posterior buys: intervals</font>](#intervals)
* [<font color='#E8800A'>Where the interval widens</font>](#extrapolation)
* [<font color='#E8800A'>What the interval does not tell you</font>](#limits)
* [<font color='#E8800A'>The settings of a Bayesian model</font>](#br-params)
* [<font color='#E8800A'>Is the grid worth running?</font>](#nested)
* [<font color='#E8800A'>The grid search</font>](#search)
* [<font color='#E8800A'>The number we stand behind</font>](#closing)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>From a point to a distribution</font> <a class="anchor" id="distribution"></a>
[Back to TOC](#toc)

Least squares returns **one** set of coefficients, the set that fits the
training cars best, and predicts every price from it. A Bayesian model keeps **a
probability density over coefficient values**: how plausible nearby sets are, given
the cars. Probabilities come from ranges of values, not individual points. Three
pieces build that density.

- The **prior** is the belief about the coefficients before any car is seen. Here
  it says that each coefficient is probably small.
- The **likelihood** is how probable the observed prices are if the coefficients
  take one particular set of values.
- The **posterior** is the belief about the coefficients after the cars are seen:
  the prior, reweighted by the likelihood.

Take one coefficient, the one on `year`. Before any car is seen, the prior makes
values near zero more plausible than values far from it. Each training car then
makes some values more plausible than others, and the more cars there are, the
narrower the posterior around the values they support.

Bayes' theorem combines the three pieces for every coefficient at once:

$$\underbrace{p(\beta \mid X, y)}_{\text{posterior}} \;\propto\;
\underbrace{p(y \mid X, \beta)}_{\text{likelihood}}
\; \underbrace{p(\beta)}_{\text{prior}} .$$

Here $\beta$ is the vector of coefficients, one per prepared column. $X$ holds the
prepared training cars, one row per car, and $y$ holds their prices. $p(\cdot)$ is
a probability density, and the bar $\mid$ reads "given". The sign $\propto$ reads
"proportional to": the two sides differ by a factor, $p(y \mid X)$, that does not
depend on $\beta$, so it changes the size of the posterior but not which values of
$\beta$ it favours.

A prediction then uses every plausible set of coefficients, each weighted by its
posterior probability, and adds the scatter of a single price around the line.
That is why it arrives with a spread rather than as a single number.

<div class="alert alert-block alert-info">

**What `BayesianRidge` assumes, stated before we fit it.**

- The target is **continuous** and the relationship is linear in the
  coefficients, the same assumption ordinary linear regression makes.
- The noise, the scatter of the prices around the line, is Gaussian with an
  unknown **precision** $\alpha$. Precision is one over the variance,
  $1/\text{variance}$, so a high precision means the prices cluster tightly
  around the line.
- The coefficients have a **zero-centred Gaussian prior** with unknown precision
  $\lambda$. In words: before seeing data, we believe coefficients are more
  likely to be small than large, and a high $\lambda$ holds them more tightly
  around zero.
- Both $\alpha$ and $\lambda$ are learned from the data rather than fixed by you.

That third assumption is a *choice*. A prior centred on
zero and shared by every coefficient says "I expect modest effects", which is
reasonable for features on a common scale and would be wrong if you had strong
outside knowledge that some coefficient is large.

**The caveat:** A posterior describes uncertainty; it does not by itself
make a prediction more accurate. The coefficients a prediction uses, the posterior
mean, are a ridge fit pulled towards zero only as hard as the prior allows. A weak
prior pulls little. The uncertainty is a claim of its own: a 95 % interval says
that 95 % of the true prices fall inside their intervals, and that share can be
counted.

</div>

In [ ]:
import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_validate, train_test_split

from course_helpers import (
    PLOT_BLUE,
    PLOT_ORANGE,
    Scoreboard,
    record,
    result_context,
    results_before,
)

random_state = 33
np.random.seed(random_state)  # For reproducibility
figsize = (7, 5)

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
regression_recipe_log = "../../logs/week_04_feature_work_regression_log.json"


__Step 1:__ Load the regression spine and split it.

In [ ]:
cars, regression_recipe = load_recipe(
    "../../data/interim/cars4you.csv",
    regression_recipe_log,
)
cars_inputs = cars.loc[:, regression_recipe["categorical"] + regression_recipe["numeric"]].copy()
y_cars = cars["price"]

cars_train, cars_test, y_cars_train, y_cars_test = train_test_split(
    cars_inputs, y_cars, test_size=0.2, random_state=random_state,
)
print(f"{len(cars_train)} training cars, {len(cars_test)} test cars")

__Step 2:__ Prepare the cars, and fix the folds every score is read on. Scaling
matters more than usual here. The prior expects every coefficient to be about the
same small size, and that only makes sense if every column is measured on a common
scale.

In [ ]:
cars_prep = FoldPreprocessor(regression_recipe)
cars_prep.fit(cars_train, y_cars_train)
X_cars_train = pd.DataFrame(
    cars_prep.transform(cars_train),
    columns=cars_prep.get_feature_names_out(),
    index=cars_train.index,
)
X_cars_test = pd.DataFrame(
    cars_prep.transform(cars_test),
    columns=cars_prep.get_feature_names_out(),
    index=cars_test.index,
)

# Ten folds of the training cars: every validation score below is a mean over them.
cars_folds = outer_splitter("regression", random_state=random_state)
cars_prepared = prepared_folds(
    FoldPreprocessor(regression_recipe), cars_folds, cars_train, y_cars_train, n_jobs=4)

# <font color='#E8800A'>Fitting it</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.BayesianRidge.html'>sklearn.linear_model.BayesianRidge(*, max_iter=300, tol=0.001, alpha_1=1e-06, alpha_2=1e-06, lambda_1=1e-06, lambda_2=1e-06, alpha_init=None, lambda_init=None, compute_score=False, fit_intercept=True, copy_X=True, verbose=False)</a>

**Definition:**
Bayesian ridge regression. Places Gamma priors on the noise precision $\alpha$ and the weight precision $\lambda$, then estimates both from the data alongside the coefficients.

**Main Parameters:**
- `lambda_1` (default `1e-6`): shape of the Gamma prior on the **weight** precision $\lambda$. Raising it pushes the coefficients towards zero.
- `alpha_1` (default `1e-6`): shape of the Gamma prior on the **noise** precision $\alpha$. Raising it insists the noise is small.
- `max_iter` (default `300`): cap on the rounds that alternate between the coefficients and the two precisions. The fit stops earlier once the coefficients change by less than `tol` (default `1e-3`).

**Main Attributes (after fitting):**
- `alpha_`: the estimated noise precision.
- `lambda_`: the estimated weight precision.
- `coef_`: the coefficients the predictions use, the posterior mean.
- `intercept_`: the intercept.
- `sigma_`: the covariance of the coefficients after the data.
- `X_offset_`: the mean of each column over the training rows.
- `n_iter_`: how many rounds the fit took.

**Main Methods:**
- `fit(X, y)`
- `predict(X, return_std=True)`: the prediction **and** its standard deviation.
</div>

__Step 3:__ Score `BayesianRidge` on the folds, and open the log.

In [ ]:
bayesian_model = PreparedEstimator(FoldPreprocessor(regression_recipe), BayesianRidge())
bayesian_cv = cross_validate(
    bayesian_model, cars_train, y_cars_train, cv=cars_folds,
    scoring="neg_mean_absolute_error", return_train_score=True)
bayesian_fitted = clone(bayesian_model).fit(cars_train, y_cars_train)

In [ ]:
cars_log = []
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "BayesianRidge()",
                  "train": -bayesian_cv["train_score"],
                  "validation": -bayesian_cv["test_score"], "model": bayesian_fitted},
       cars_test, y_cars_test, metric=mean_absolute_error, label="MAE")

**MAE 3,034.97 EUR on the validation folds and 3,098.08 on the test cars.**
The train MAE, 2,997.85, sits just below the validation MAE: the model fits the cars
it was fitted on hardly better than those it never saw.

# <font color='#E8800A'>What the model learned</font> <a class="anchor" id="learned"></a>
[Back to TOC](#toc)

Fitting finds the coefficients and the two precisions together. Each step
in this section reads one fitted attribute.

__Step 4:__ Fit `BayesianRidge` on the prepared training cars.

In [ ]:
bayesian = BayesianRidge().fit(X_cars_train, y_cars_train)

__Step 5:__ Read the two precisions. `alpha_` is the noise precision: how tightly
the training prices cluster around the fitted line. One over its square root,
$1/\sqrt{\alpha}$, is a standard deviation in euros: how far a price typically falls
from the line. `lambda_` is the weight precision: how tightly the fitted prior holds
the coefficients around zero, and $1/\sqrt{\lambda}$ is the typical size that prior
expects of a coefficient. Print each with the standard deviation it implies.

In [ ]:
print(f"alpha_  = {bayesian.alpha_:.2e}   noise sd  1 / sqrt(alpha_)  ="
      f" {1 / np.sqrt(bayesian.alpha_):9,.2f} EUR")
print(f"lambda_ = {bayesian.lambda_:.2e}   prior sd  1 / sqrt(lambda_) ="
      f" {1 / np.sqrt(bayesian.lambda_):9,.2f} EUR")

**A price typically falls 4,890.61 EUR from the fitted line, and the fitted
prior expects a coefficient of about 14,966.06 EUR.** The precisions themselves are
tiny numbers, 4.18e-08 and 4.46e-09, because each is one over a variance in squared
euros.

__Step 6:__ Read the coefficients. `coef_` holds one coefficient per prepared column. It
is the posterior mean, the value the prediction uses. `sigma_` is the covariance of
the coefficients after the data: the square root of its diagonal is each
coefficient's posterior standard deviation, how uncertain that coefficient still
is. `intercept_` is the price the line gives a car whose prepared columns are all
zero. Tabulate each coefficient beside its posterior standard deviation, and print
the intercept.

In [ ]:
coefficients = pd.DataFrame({"coef_": bayesian.coef_,
                             "posterior sd": np.sqrt(np.diag(bayesian.sigma_))},
                            index=X_cars_train.columns)
print(f"intercept_ = {bayesian.intercept_:,.2f} EUR")
coefficients.round(2)

**The posterior standard deviations run from 269.94 for
`transmission_Manual` to 4,700.97 for `fuelType_Electric`, all below the prior's
14,966.06.** The training cars narrowed the belief about every coefficient, some far
more than others.

# <font color='#E8800A'>The prior is a ridge penalty</font> <a class="anchor" id="prior-is-penalty"></a>
[Back to TOC](#toc)

Least squares picks the coefficients that make the sum of squared errors
over the training cars as small as possible:

$$\min_\beta \; \sum_i \big(y_i - x_i^\top \beta\big)^2 .$$

For this derivation, subtract the training mean from every prepared column and
from the prices. Here $x_i$ is car $i$'s centred row and $y_i$ its price minus the
training mean. The prediction $x_i^\top\beta$ is a deviation from that mean.
Centring lets the equations omit an intercept; in the original units it is
$\bar y-\bar x^\top\beta$, where the bars denote training means. Ridge leaves this
intercept unpenalised.

**Ridge regression** adds a penalty: a constant $k$ times the sum of the squared
coefficients, $k \sum_j \beta_j^2$, where $\beta_j$ is the coefficient of prepared
column $j$. A large coefficient now costs extra, so the fit pulls every coefficient
towards zero, and the larger $k$, the harder the pull. `Ridge(alpha=...)` names this
constant `alpha`, which is not the noise precision $\alpha$ of this notebook.

A prior centred on zero says the same thing in probability: large coefficients are
unlikely. The Gaussian prior with precision $\lambda$, and the Gaussian noise with
precision $\alpha$, give

$$p(\beta) \propto \exp\Big(-\frac{\lambda}{2}\sum_j \beta_j^2\Big), \qquad
p(y \mid X, \beta) \propto \exp\Big(-\frac{\alpha}{2}\sum_i \big(y_i - x_i^\top\beta\big)^2\Big),$$

where $\exp(z)$ is $e^z$. Each falls as its sum grows.

The most probable coefficients are those at which the posterior, the likelihood
times the prior, peaks. The logarithm grows with its argument, so the posterior
peaks where minus its logarithm is smallest, and minus the logarithm of
$\exp(-z)$ is $z$. Minus the logarithm of the posterior is therefore

$$\frac{\alpha}{2}\sum_i \big(y_i - x_i^\top\beta\big)^2 + \frac{\lambda}{2}\sum_j \beta_j^2 ,$$

plus a term that does not depend on $\beta$. Dividing it by $\alpha/2$ changes its
size but not where it is smallest, and leaves the ridge objective with the constant
$k = \lambda/\alpha$.

<div class="alert alert-block alert-info">

**The most probable coefficients are the ridge coefficients.**

$$\underbrace{\arg\max_\beta \; p(\beta \mid X, y)}_{\text{Bayesian}}
\;=\;
\underbrace{\arg\min_\beta \; \lVert y - X\beta \rVert^2
+ \frac{\lambda}{\alpha}\lVert \beta \rVert^2}_{\text{ridge}} .$$

On the left, $\arg\max_\beta$ names the $\beta$ at which the posterior peaks. On the
right, $\arg\min_\beta$ names the $\beta$ at which the penalised error is smallest.
$\lVert y - X\beta \rVert^2$ is the sum of squared errors over the training cars,
and $\lVert \beta \rVert^2 = \sum_j \beta_j^2$.

The ridge constant is $\lambda/\alpha$, the prior precision divided by the noise
precision. A **tight** prior (large $\lambda$) or **noisy** data (small $\alpha$)
both push the coefficients harder towards zero.

</div>

__Step 7:__ Compute the ridge constant the fitted precisions imply, `lambda_ / alpha_`.

In [ ]:
implied_penalty = bayesian.lambda_ / bayesian.alpha_
print(f"implied ridge constant lambda_ / alpha_ = {implied_penalty:.4f}")

__Step 8:__ Fit plain `Ridge` with that constant, and compare its coefficients with the Bayesian ones.

In [ ]:
equivalent_ridge = Ridge(alpha=implied_penalty).fit(X_cars_train, y_cars_train)
print("largest coefficient difference from Ridge:"
      f" {np.abs(bayesian.coef_ - equivalent_ridge.coef_).max():.2e} EUR")

**The constant is 0.1068, and the largest coefficient difference is 6.97e-10
EUR.** `Ridge` with the implied constant reproduces the Bayesian coefficients to
rounding error, so the two fits are the same arithmetic. The data set that constant
while `BayesianRidge` fitted, with no search over penalties.

# <font color='#E8800A'>One car's prediction and its range</font> <a class="anchor" id="one-car"></a>
[Back to TOC](#toc)

A prediction is the intercept plus each prepared column's value times its
coefficient:

$$\hat{y} = \beta_0 + \sum_j x_j \beta_j ,$$

where $x_j$ is the car's value in prepared column $j$, $\beta_j$ is that column's
coefficient in `coef_`, and $\beta_0$ is `intercept_`.

__Step 9:__ Predict the first test car by hand, beside `predict`. In the code, `x @ bayesian.coef_` multiplies each column's value by its coefficient and adds the products.

In [ ]:
car = X_cars_test.iloc[[0]]
x = car.to_numpy()[0]
prediction_by_hand = bayesian.intercept_ + x @ bayesian.coef_
print(f"by hand: {prediction_by_hand:,.2f} EUR   predict: {bayesian.predict(car)[0]:,.2f} EUR")
print(f"its true price: {y_cars_test.iloc[0]:,.0f} EUR")

**11,612.19 EUR, by hand and from `predict`**, for a car that sold for
13,500 EUR.

The spread has two parts. The first is the noise: how far a price scatters
around the line even if the coefficients were known exactly. Its variance is
$1/\alpha$, the same for every car. The second is the uncertainty about the
coefficients: how much the plausible sets of coefficients disagree about this car's
price. Its variance is $c^\top \Sigma\, c$, where $\Sigma$ is `sigma_` and
$c = x - \bar{x}$ is the car's row minus $\bar{x}$, the mean of each prepared column
over the training cars, stored in `X_offset_`. The product multiplies $\Sigma$ by
$c$ on both sides, `c @ bayesian.sigma_ @ c` in code, and the farther the car sits
from the training means, the larger it grows. The standard deviation adds the two
variances and takes the square root:

$$\sigma(x) = \sqrt{\frac{1}{\alpha} + c^\top \Sigma\, c} .$$

Scikit-learn 1.9.0 fits the coefficients on columns centred on their training means,
so `predict` centres the row the same way, through `X_offset_`, before this sum.

__Step 10:__ Compute the first test car's standard deviation by hand, part by part, beside `predict(car, return_std=True)`.

In [ ]:
c = x - bayesian.X_offset_
noise_part = 1 / bayesian.alpha_
coefficient_part = c @ bayesian.sigma_ @ c
std_by_hand = np.sqrt(noise_part + coefficient_part)
# One standard deviation per car; .item() reads the single one.
_, spread = bayesian.predict(car, return_std=True)
print(f"noise part        1 / alpha_      = {noise_part:12,.0f} squared EUR")
print(f"coefficient part  c @ sigma_ @ c  = {coefficient_part:12,.0f} squared EUR")
print(f"sd by hand: {std_by_hand:,.2f} EUR   predict: {spread.item():,.2f} EUR")

**4,899.92 EUR, by hand and from `predict`.** The noise part, 23,918,110
squared euros, is the same for every car. The coefficient part, 91,098, belongs to
this car and grows as a car moves away from the training means. For this car the
noise part is almost the whole spread.

# <font color='#E8800A'>What the posterior buys: intervals</font> <a class="anchor" id="intervals"></a>
[Back to TOC](#toc)

`predict(X, return_std=True)` returns two numbers per car: the prediction, and
the standard deviation the one-car steps computed by hand. The predictive
distribution is normal, and 95 % of a normal distribution lies within 1.96 standard
deviations of its mean. The prediction plus or minus 1.96 standard deviations is
therefore a **nominal** 95 % interval: 95 % is what the model claims. Its
**coverage** is the share of true prices that actually fall inside.

__Step 11:__ Turn each validation prediction's spread into an interval, and count the validation prices inside. Each fold's model predicts its own validation part.

In [ ]:
# Each fold's model predicts its validation part with a spread, so every
# training car is predicted twice, once per repeat.
fold_predictions = [
    BayesianRidge().fit(fold["X_fit"], fold["y_fit"]).predict(fold["X_score"], return_std=True)
    for fold in cars_prepared
]
mean_prediction = np.concatenate([mean for mean, _ in fold_predictions])
prediction_std = np.concatenate([spread for _, spread in fold_predictions])
actual_price = np.concatenate([fold["y_score"] for fold in cars_prepared])
lower = mean_prediction - 1.96 * prediction_std
upper = mean_prediction + 1.96 * prediction_std
inside = ((actual_price >= lower) & (actual_price <= upper)).mean()

print(f"interval half-width runs {1.96 * prediction_std.min():,.0f} EUR"
      f" to {1.96 * prediction_std.max():,.0f} EUR")
print(f"validation prices inside the nominal 95% interval: {inside:.1%}")

__Step 12:__ Do the same for the test cars, with the model fitted on all the training cars.

In [ ]:
test_mean, test_std = bayesian.predict(X_cars_test, return_std=True)
test_inside = (np.abs(y_cars_test.to_numpy() - test_mean) <= 1.96 * test_std).mean()
print(f"test prices inside the nominal 95% interval: {test_inside:.1%}")

<div class="alert alert-block alert-warning">

**96.2 % of the validation prices and 96.2 % of the test prices fall inside
their nominal 95 % intervals.** The model claimed a coverage and delivered it,
slightly generously, on cars it had not seen.

The intervals are wide. Each reaches **8,803 to 11,152 EUR** either side of its
prediction, against a validation MAE of 3,034.97 for the same model.

</div>

__Step 13:__ Draw the interval around every validation prediction, with the true prices.

In [ ]:
order = np.argsort(mean_prediction)
plt.figure(figsize=figsize)
plt.fill_between(range(len(order)), lower[order], upper[order],
                 color=PLOT_BLUE, alpha=0.25, label="95% interval")
plt.plot(range(len(order)), mean_prediction[order], color=PLOT_BLUE,
         label="prediction")
plt.scatter(range(len(order)), actual_price[order], color=PLOT_ORANGE, s=4,
            alpha=0.4, label="actual price")
plt.xlabel("validation predictions, ordered by prediction")
plt.ylabel("price (EUR)")
plt.legend()
plt.title("Every prediction arrives with a range")
plt.show()

# <font color='#E8800A'>Where the interval widens</font> <a class="anchor" id="extrapolation"></a>
[Back to TOC](#toc)

A model in use is asked about cars unlike any it was trained on. Its
prediction for such a car is a guess outside the data, and this section asks whether
the interval warns about that guess by widening.

The one-car steps split the spread into two parts. The noise part, $1/\alpha$, is
the same for every car. The coefficient part, $c^\top \Sigma\, c$, grows as the car
moves away from the training means, because $c$ is the car's distance from them. A
car with an old enough `year` should therefore get a wider interval.

The probe takes one typical training car, the one whose mileage is closest to the
median, and moves its `year` back. A **span** is the range of years among the
training cars, from the oldest to the newest. The car is moved three spans, then ten
spans, before the oldest training year. Read the spread and the predicted price
together: the spread says how uncertain the model is, and the price says whether the
answer still describes a car.

__Step 14:__ Build the three probe cars.

In [ ]:
typical_index = cars_train["mileage"].sub(cars_train["mileage"].median()).abs().idxmin()
typical = cars_train.loc[[typical_index]]
oldest = cars_train["year"].min()
span = cars_train["year"].max() - oldest
print(f"training years run {oldest} to {cars_train['year'].max()}: a span of {span} years")
probes = pd.concat([typical,
                    typical.assign(year=oldest - 3 * span),
                    typical.assign(year=oldest - 10 * span)])
probes.index = ["typical car", "3 spans before the oldest", "10 spans before the oldest"]
probes

__Step 15:__ Predict the three probe cars, each with its standard deviation.

In [ ]:
probes_ready = pd.DataFrame(cars_prep.transform(probes), columns=X_cars_train.columns,
                            index=probes.index)
probe_mean, probe_std = bayesian.predict(probes_ready, return_std=True)
for label, mean, spread in zip(probes.index, probe_mean, probe_std):
    print(f"{label:28s} {mean:11,.0f} EUR  +/- {spread:8,.0f}")

**The spread grows from ±4,909 to ±6,551, then to ±13,094 EUR.** Far from
the training cars, many sets of coefficients remain plausible, and they disagree
about a car like this.

**The prediction becomes impossible.** It falls from 16,610 EUR to -351,642 EUR, a
negative price, because a straight line keeps going outside the range it was fitted
on. A ±13,094 band around -351,642 holds only negative prices. `LinearRegression`
has no `return_std`: least squares returns a prediction with no spread at all.

The width measures how much the plausible coefficients disagree. Nothing in it
checks whether the input describes a car that can exist.

# <font color='#E8800A'>What the interval does not tell you</font> <a class="anchor" id="limits"></a>
[Back to TOC](#toc)

Inside the training range, could the width decide which predictions to
trust? If wider intervals go with larger errors, the width ranks the predictions,
and a user could check the widest ones first. The steps in this section measure that
link, check that it is more than chance, and measure how much the width varies from
car to car.

The link is measured by the **correlation** between each prediction's interval width
and its absolute error, the distance between the prediction and the true price. A
correlation runs from -1 to +1. Near +1, the wider the interval, the larger the
error; near 0, the width says nothing about the error.

The correlation is computed inside each fold, and the ten are averaged. Each fold is
one fitted model with its own estimate of the noise, so every interval in one fold
shares the same noise part and differs from the others only through its coefficient
part.

__Step 16:__ Correlate interval width with absolute error inside each fold, and average the ten correlations.

In [ ]:
# One fold is one fitted model, so its widths differ only through the
# coefficient part of the spread.
fold_widths = [spread for _, spread in fold_predictions]
fold_errors = [np.abs(fold["y_score"].to_numpy() - mean)
               for fold, (mean, _) in zip(cars_prepared, fold_predictions)]
observed = np.mean([np.corrcoef(width, error)[0, 1]
                    for width, error in zip(fold_widths, fold_errors)])
print(f"correlation of width and error inside a fold, mean of ten: {observed:+.4f}")

__Step 17:__ Pool the ten folds into one correlation, to see why the comparison stays inside a fold.

In [ ]:
pooled = np.corrcoef(prediction_std, np.abs(actual_price - mean_prediction))[0, 1]
print(f"correlation of width and error, pooled across the ten folds: {pooled:+.4f}")

__Step 18:__ Compute the same correlation on the test cars, which one model predicts.

In [ ]:
test_error = np.abs(y_cars_test.to_numpy() - test_mean)
print("correlation of width and error on the test cars:"
      f" {np.corrcoef(test_std, test_error)[0, 1]:+.4f}")

**+0.1872 inside a fold and +0.1196 on the 800 test cars, but -0.0080 pooled
across the ten folds.** Within one model, wider intervals go with larger errors,
weakly. Pooled, the link disappears, because across folds the widths mostly say which
model made the prediction.

__Step 19:__ Split each fold at its median width, and compare the MAE of the narrow half with the MAE of the wide half.

In [ ]:
narrow = np.concatenate([error[width <= np.median(width)]
                         for width, error in zip(fold_widths, fold_errors)])
wide = np.concatenate([error[width > np.median(width)]
                       for width, error in zip(fold_widths, fold_errors)])
print(f"MAE where the interval is narrow: {narrow.mean():,.2f} EUR")
print(f"MAE where the interval is wide:   {wide.mean():,.2f} EUR")

**2,449.62 EUR where the interval is narrow, 3,620.32 where it is wide.** In
euros, the cars in the wide half are missed by more.

__Step 20:__ A correlation of +0.1872 could still come from chance. The **shuffle test** measures how easily: if the width carried no information about the error, shuffling the widths among the cars of a fold would often reach the observed correlation, so the share of 1,000 shuffles that reach it measures how easily chance does. The textbook test of a correlation instead assumes both quantities are roughly normal, so first print the smallest and the largest absolute error.

In [ ]:
print(f"absolute errors run {min(error.min() for error in fold_errors):,.0f}"
      f" to {max(error.max() for error in fold_errors):,.0f} EUR")

**1 to 83,616 EUR**, against an MAE near 3,000: a long right tail, far from
normal. A shuffle needs no assumption about that shape.

__Step 21:__ Shuffle the widths inside each fold 1,000 times, and count the shuffles whose mean correlation is at least as large as the observed one.

In [ ]:
# Shuffling inside each fold keeps every fold's own noise level.
shuffler = np.random.default_rng(random_state)
at_least_as_large = 0
for _ in range(1000):
    shuffled = np.mean([np.corrcoef(shuffler.permutation(width), error)[0, 1]
                        for width, error in zip(fold_widths, fold_errors)])
    if abs(shuffled) >= abs(observed):
        at_least_as_large += 1
print(f"chance reproduced it in {at_least_as_large} of 1,000 shuffles")

**0 of 1,000 shuffled pairings matched the observed mean correlation.**
This supports the observed link as an exploratory check. Each car appears in two
validation folds, and these shuffles do not preserve that dependence, so the count
is not a formal significance test. Within one model the
interval does identify the harder cars.

__Step 22:__ Measure how much the width varies inside a fold: the widest interval divided by the narrowest.

In [ ]:
print("widest over narrowest interval within a fold: at most"
      f" {max(width.max() / width.min() for width in fold_widths):.3f}")

<div class="alert alert-block alert-warning">

**Within a fold the widest interval is at most 1.152 times the narrowest.**
Only the coefficient part of the spread differs between the cars of one fold, and
with thousands of training cars it is small beside the noise part, as the first test
car showed. The width ranks the cars, but while the errors run from 1 to 83,616 EUR
the width moves by at most 15.2 %: read as a per-car error bar, it would understate
how far apart the errors are.

</div>

# <font color='#E8800A'>The settings of a Bayesian model</font> <a class="anchor" id="br-params"></a>
[Back to TOC](#toc)

`BayesianRidge` learns the two precisions, `alpha_` and `lambda_`, from the
data, so neither is a setting. What you set instead is a belief about each precision
before any car is seen: a prior on the precision, called a **hyperprior**. Each
hyperprior is a Gamma distribution, a distribution over positive numbers described
by a shape and a rate, and its mean is the shape divided by the rate. `lambda_1` and
`lambda_2` are the shape and the rate of the hyperprior on $\lambda$; `alpha_1` and
`alpha_2` are those of the hyperprior on $\alpha$.

This section moves the two shapes, `lambda_1` and `alpha_1`, and the iteration
budget, `max_iter`, each alone with the others at their defaults, to show what each
does. The choice is made by one grid search over every combination of the values
tried; in a project you would go straight to that grid.

**`lambda_1`: how strongly you insist the coefficients are small.** It is
the shape of the hyperprior on $\lambda$, the weight precision. Raising it raises that
hyperprior's mean, `lambda_1 / lambda_2`, so before any car is seen it asserts that
$\lambda$ is large, which means the coefficients sit near zero.

The default is `1e-6`, and the `lambda_` the fit learned is 4.46e-09: the 3,200
training cars currently overrule the hyperprior. **Prediction:** nothing changes
while the hyperprior stays weaker than the data; once it is stronger, the
coefficients do not shrink gradually, they go to zero together.

__Step 23:__ Set the values of `lambda_1` to try: `[1e-9, 1e-6, 1e-4, 1e-3, 1e-2, 1.0]`. At `1e-6` it is at its default.

In [ ]:
lambda_values = ...

In the loop, the model is `BayesianRidge(lambda_1=value)`. Each value
changes only the shape of the hyperprior on $\lambda$, and the sweep shows the value
at which that hyperprior starts to outweigh the training cars.

In [ ]:
lambda_rows = []
for value in lambda_values:
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
    fitted = clone(model).fit(X_cars_train, y_cars_train)
    validation = [mean_absolute_error(fold["y_score"], trained.predict(fold["X_score"]))
                  for fold, trained in zip(cars_prepared, fold_models)]
    lambda_rows.append({
        "configuration": f"BayesianRidge(lambda_1={value:g})", "lambda_1": value,
        "train": [mean_absolute_error(fold["y_fit"], trained.predict(fold["X_fit"]))
                  for fold, trained in zip(cars_prepared, fold_models)],
        "validation": validation, "cv_MAE": np.mean(validation),
        "test_MAE": mean_absolute_error(y_cars_test, fitted.predict(X_cars_test)),
        "learned_lambda": fitted.lambda_, "largest_coef": np.abs(fitted.coef_).max(),
        "model": model,
    })
lambda_sweep = pd.DataFrame(lambda_rows)
lambda_sweep[["lambda_1", "cv_MAE", "test_MAE", "learned_lambda", "largest_coef"]].round(
    {"cv_MAE": 2, "test_MAE": 2})

In [ ]:
# The sweep's winner joins the log.
best = lambda_sweep.loc[lambda_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

<div class="alert alert-block alert-warning">

**From 1e-9 to 1e-3 the validation MAE stays at 3,034.97; at 1e-2 it jumps
to 7,231.10.** Up to 1e-3 the learned $\lambda$ stays about **4.46e-09** and the
largest coefficient about **4.40e+04**: the hyperprior changes nothing the table
shows. The lowest validation MAE joins the log: `BayesianRidge(lambda_1=0.001)`, at
3,034.9682 against the default's 3,034.9685.

At 1e-2 the learned $\lambda$ jumps to **9,999**, and to **1e+06** at 1.0, close to
the hyperprior's means, `1e-2 / 1e-6` and `1.0 / 1e-6`. The largest coefficient
falls to **8.38e-06**. With every coefficient near zero, the model predicts its
intercept, the training mean, for every car: **7,231.10** on the folds and
**6,843.46** on the test cars.

The switch from no effect to no coefficients falls inside one decade, and the sweep
tests no value between them.

</div>

**`alpha_1`: how strongly you insist the noise is small.** It is the shape
of the hyperprior on $\alpha$, the noise precision. Raising it asserts, before any
car is seen, that the prices sit tightly around the line.

The coefficients feel $\alpha$ only through the ridge constant $\lambda/\alpha$,
which the data set at 0.1068. A larger $\alpha$ shrinks that constant and brings the
fit closer to least squares, so the predictions should barely move. The interval
should move a great deal: its noise part, $1/\alpha$, was almost the whole of the
first test car's spread. **Prediction:** the MAE nearly unchanged, the intervals far
too narrow.

__Step 24:__ Set the values of `alpha_1` to try: `[1e-9, 1e-6, 1.0, 1e3, 1e5, 1e7]`. At `1e-6` it is at its default.

In [ ]:
alpha_values = ...

In the loop, the model is `BayesianRidge(alpha_1=value)`. Each value
changes only the shape of the hyperprior on $\alpha$, and the sweep prints the MAE
beside the mean predictive standard deviation and the coverage, to show which of
them the setting moves.

In [ ]:
alpha_rows = []
for value in alpha_values:
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
    spreads = [trained.predict(fold["X_score"], return_std=True)
               for fold, trained in zip(cars_prepared, fold_models)]
    test_mean_value, test_spread = clone(model).fit(X_cars_train, y_cars_train).predict(
        X_cars_test, return_std=True)
    validation = [mean_absolute_error(fold["y_score"], mean)
                  for fold, (mean, _) in zip(cars_prepared, spreads)]
    alpha_rows.append({
        "configuration": f"BayesianRidge(alpha_1={value:g})", "alpha_1": value,
        "train": [mean_absolute_error(fold["y_fit"], trained.predict(fold["X_fit"]))
                  for fold, trained in zip(cars_prepared, fold_models)],
        "validation": validation, "cv_MAE": np.mean(validation),
        "test_MAE": mean_absolute_error(y_cars_test, test_mean_value),
        "cv_mean_sd": np.mean([spread.mean() for _, spread in spreads]),
        "cv_coverage": np.mean([(np.abs(fold["y_score"] - mean) <= 1.96 * spread).mean()
                                for fold, (mean, spread) in zip(cars_prepared, spreads)]),
        "test_coverage": (np.abs(y_cars_test - test_mean_value) <= 1.96 * test_spread).mean(),
        "model": model,
    })
alpha_sweep = pd.DataFrame(alpha_rows)
alpha_sweep[["alpha_1", "cv_MAE", "test_MAE", "cv_mean_sd", "cv_coverage",
             "test_coverage"]].round({"cv_MAE": 2, "test_MAE": 2, "cv_mean_sd": 1,
                                      "cv_coverage": 4, "test_coverage": 4})

In [ ]:
# The sweep's winner joins the log.
best = alpha_sweep.loc[alpha_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

<div class="alert alert-block alert-warning">

**The validation MAE moves from 3,034.97 to 3,038.16, while the mean
predictive standard deviation falls from 4,898.5 to 55.2.** A large noise precision
shrinks the ridge constant `lambda_ / alpha_` towards zero, so the fit approaches
least squares, and the MAE moves by about three euros.

The coverage falls with the spread, from **0.9625** to **0.0292** on the folds and
**0.0350** on the test cars: the last model's "95 %" intervals hold about 3 % of the
prices.

The three smallest values tie at 3,034.97, and the lowest validation MAE joins the
log: `BayesianRidge(alpha_1=1e-09)`, at 3,034.9685, the default's own value to four
decimals.

**The MAE column cannot show this failure.** MAE reads only the predictions; the
coverage columns show the intervals failing.

</div>

**`max_iter`: how many rounds the fit may take.** `BayesianRidge` cannot
compute the coefficients and the two precisions in one go, so it alternates. Given
the current precisions, it fits the coefficients; given the coefficients, it
re-estimates the precisions. It starts from $\lambda = 1$ and $\alpha$ equal to one
over the variance of the training prices: a ridge constant $\lambda/\alpha$ as large
as that variance, so its first coefficients sit near zero. `max_iter` caps the
rounds, and the fit stops earlier once the coefficients change by less than `tol`.

This is a budget rather than a belief, but a budget set too low still returns a
fitted model, with no error. **Prediction:** after one or two rounds the model should
look like the collapsed end of the `lambda_1` sweep.

__Step 25:__ Set the iteration budgets to try: `[1, 2, 3, 5, 10, 300]`. At `300` it is at its default.

In [ ]:
iter_values = ...

In the loop, the model is `BayesianRidge(max_iter=value)`. Each value
changes only how many rounds the fit may take, and the sweep shows what a fit stopped
early predicts and, through `n_iter_`, where each fit stopped.

In [ ]:
iter_rows = []
for value in iter_values:
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in cars_prepared]
    fitted = clone(model).fit(X_cars_train, y_cars_train)
    validation = [mean_absolute_error(fold["y_score"], trained.predict(fold["X_score"]))
                  for fold, trained in zip(cars_prepared, fold_models)]
    iter_rows.append({
        "configuration": f"BayesianRidge(max_iter={value})", "max_iter": value,
        "train": [mean_absolute_error(fold["y_fit"], trained.predict(fold["X_fit"]))
                  for fold, trained in zip(cars_prepared, fold_models)],
        "validation": validation, "cv_MAE": np.mean(validation),
        "test_MAE": mean_absolute_error(y_cars_test, fitted.predict(X_cars_test)),
        "stopped_after": fitted.n_iter_, "learned_lambda": fitted.lambda_,
        "model": model,
    })
iter_sweep = pd.DataFrame(iter_rows)
iter_sweep[["max_iter", "cv_MAE", "test_MAE", "stopped_after", "learned_lambda"]].round(
    {"cv_MAE": 2, "test_MAE": 2})

In [ ]:
# The sweep's winner joins the log.
best = iter_sweep.loc[iter_sweep["cv_MAE"].idxmin()]
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"],
                  "model": clone(best["model"]).fit(X_cars_train, y_cars_train)},
       X_cars_test, y_cars_test, "one setting changed", metric=mean_absolute_error,
       label="MAE")

**7,220.90 after one round, 5,814.66 after two, 3,105.30 after three and
3,034.92 after five.** One round leaves the model near the 7,231.10 the `lambda_1`
sweep printed for a model with no coefficients: its `learned_lambda` is still
2.97e-03, against 4.46e-09 once the fit has converged.

The `stopped_after` column, read from `n_iter_`, shows each fit at its cap for every
budget up to five, and at 10 for budgets of 10 and 300: left alone, the fit
converges after 10 rounds, so the default budget of 300 is never reached. Five
rounds score 5 cents below the default on the folds, and that row joins the log.

# <font color='#E8800A'>Is the grid worth running?</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

Combining the three sweep winners would be a guess at the best joint setting.
A grid fits every combination of the values the sweeps tried instead, 216 here, six
values for each of the three settings, so it can land on a pairing no single sweep
visited.

Before the grid runs on all the training cars, nested cross-validation estimates
what that procedure is worth. It runs the whole grid inside each of the ten outer
folds: inner folds, cut from the outer training part alone, choose the settings, and
the outer validation part, which took no part in that choice, scores the choice.
Nested cross-validation returns no model. Its mean estimates the error of the grid
procedure itself, and it is read against the default's validation MAE on the same
folds.

__Step 26:__ Define the grid.

In [ ]:
cars_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(regression_recipe), BayesianRidge()),
    {"estimator__lambda_1": lambda_values, "estimator__alpha_1": alpha_values,
     "estimator__max_iter": iter_values},
    scoring="neg_mean_absolute_error", cv=cars_folds, n_jobs=4,
)

__Step 27:__ Estimate the grid with nested cross-validation.

In [ ]:
cars_nested = cross_validate(
    cars_template, cars_train, y_cars_train, cv=cars_folds,
    scoring={"mae": "neg_mean_absolute_error",
             "rmse": "neg_root_mean_squared_error"},
    return_train_score=True,
)
cars_nested_rows = []
for metric in ("mae", "rmse"):
    # The scorers return negated errors, so the sign is flipped back.
    validation_values = -cars_nested[f"test_{metric}"]
    cars_nested_rows.append({
        "metric": metric,
        "outer_train_mean": -cars_nested[f"train_{metric}"].mean(),
        "outer_validation_mean": validation_values.mean(),
        "fold_sd": validation_values.std(ddof=1),
    })
print(pd.DataFrame(cars_nested_rows).round(2).to_string(index=False))

**MAE 3,016.00 EUR over the ten outer folds, with a fold standard deviation of
163.62 EUR.** That spread describes variation across the folds. The outer
training mean, 2,985.73 EUR, sits close to the validation mean. Because the repeated
folds share cars, their spread is not an independent-sample standard error or proof
that overfitting is absent.

__Step 28:__ Set the default's validation MAE on the same folds beside the nested mean, compare the paired errors in euros.

In [ ]:
default_validation = -bayesian_cv["test_score"].mean()
paired_difference = -cars_nested["test_mae"] + bayesian_cv["test_score"]
difference = paired_difference.mean()
print(f"the default on the same folds: MAE {default_validation:,.2f} EUR")
print(f"nested minus default: mean {difference:+,.2f} EUR;"
      f" fold range {paired_difference.min():+,.2f} to {paired_difference.max():+,.2f} EUR;"
      f" grid lower in {(paired_difference < 0).sum()} of {len(paired_difference)} folds")

**The nested mean sits 18.97 EUR below the default's 3,034.97.** The paired
fold differences show whether that small average gain holds across the same splits.
They describe this run; overlapping folds do not establish a significant gain. But nested
cross-validation estimates the grid procedure and returns no model, so the model the
week deploys is the grid's winner, refitted on all the training cars. The number the
board carries for it is the nested estimate.

# <font color='#E8800A'>The grid search</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

__Step 29:__ Run the grid on all the training cars.

In [ ]:
cars_search = clone(cars_template).fit(cars_train, y_cars_train)

__Step 30:__ Rank the combinations, and refit each of the five best to read its test MAE beside its validation MAE.

In [ ]:
# The five best combinations, each beside the test MAE of its refit.
results = cars_search.cv_results_
ranking = pd.DataFrame({
    "lambda_1": [f"{value:g}" for value in results["param_estimator__lambda_1"]],
    "alpha_1": [f"{value:g}" for value in results["param_estimator__alpha_1"]],
    "max_iter": results["param_estimator__max_iter"],
    "cv_MAE": -results["mean_test_score"],
}).nsmallest(5, "cv_MAE")
ranking["test_MAE"] = [
    mean_absolute_error(y_cars_test, clone(cars_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(
            cars_train, y_cars_train).predict(cars_test))
    for index in ranking.index]
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(2).to_string())

**The grid picks `lambda_1=1e-4, alpha_1=1e5, max_iter=2`, at 3,015.29 on the
validation folds and 3,069.30 on the test cars.** No sweep visited that combination:
stopped after two rounds on its own, the model scored 5,814.66, and the winner pairs
that budget with a noise hyperprior that barely moved the MAE alone. The three sweep
winners combined, `lambda_1=0.001, alpha_1=1e-09, max_iter=5`, come fourth, at
3,027.86.

__Step 31:__ Log the winner's own score on the folds.

In [ ]:
# The winner's own score is the best of all of them on the same folds.
winner_cars = cross_validate(
    clone(cars_search.best_estimator_), cars_train, y_cars_train,
    cv=cars_folds, scoring="neg_mean_absolute_error", return_train_score=True)
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "joint search winner",
                  "train": -winner_cars["train_score"],
                  "validation": -winner_cars["test_score"],
                  "model": cars_search.best_estimator_},
       cars_test, y_cars_test, "winner's own score", metric=mean_absolute_error, label="MAE")

__Step 32:__ Log the nested estimate, beside the same deployed model's test MAE.

In [ ]:
record(cars_log, {"frame": type(cars_folds).__name__, "rows": len(y_cars_train),
                  "folds": cars_folds.get_n_splits(), "name": "joint search winner",
                  "train": -cars_nested["train_mae"],
                  "validation": -cars_nested["test_mae"],
                  "model": cars_search.best_estimator_},
       cars_test, y_cars_test, "nested estimate", metric=mean_absolute_error, label="MAE")

**The winner's own validation MAE, 3,015.29, sits within a euro of its nested
estimate, 3,016.00.** Its own score is the best of 216 on the same folds, which tends
to flatter it, and the board publishes the nested estimate.

__Step 33:__ Read the winner's interval beside the default's: the grid chose on MAE, which cannot see an interval.

In [ ]:
winner = {name.split("__")[-1]: value for name, value in cars_search.best_params_.items()}
for label, settings in [("default", {}), ("search winner", winner)]:
    spreads = [BayesianRidge(**settings).fit(fold["X_fit"], fold["y_fit"]).predict(
        fold["X_score"], return_std=True) for fold in cars_prepared]
    coverage = np.mean([(np.abs(fold["y_score"] - mean) <= 1.96 * spread).mean()
                        for fold, (mean, spread) in zip(cars_prepared, spreads)])
    test_mean_value, test_spread = BayesianRidge(**settings).fit(
        X_cars_train, y_cars_train).predict(X_cars_test, return_std=True)
    print(f"{label:14s} mean sd {np.mean([spread.mean() for _, spread in spreads]):8.1f}"
          f"   95% coverage: validation {coverage:.1%},"
          f" test {(np.abs(y_cars_test - test_mean_value) <= 1.96 * test_spread).mean():.1%}")

**The winner's 95 % intervals hold 52.8 % of the validation prices and
54.9 % of the test prices**, against 96.2 % of both for the default. Its mean
predictive standard deviation is 1,126.9 EUR, against the default's 4,898.5. The grid
scored only MAE, through `scoring="neg_mean_absolute_error"`.

# <font color='#E8800A'>The number we stand behind</font> <a class="anchor" id="closing"></a>
[Back to TOC](#toc)

__Step 34:__ Report the winner and the default on the test cars.

In [ ]:
chosen_bayesian = cars_search.best_estimator_
tuned_mae = mean_absolute_error(y_cars_test, chosen_bayesian.predict(cars_test))
default_bayesian = PreparedEstimator(FoldPreprocessor(regression_recipe),
                                     BayesianRidge()).fit(cars_train, y_cars_train)
default_mae = mean_absolute_error(y_cars_test, default_bayesian.predict(cars_test))

**On the 800 held-out cars the winner scores 3,069.30 EUR and the default
3,098.08**, the test MAE of their rows in the log. Neither number is adjusted: the
configuration was chosen using validation scores alone. The test scores displayed during the sweeps did not select it.

__Step 35:__ Take the default's cross-validated MAE, on the same folds, as the baseline.

In [ ]:
# The default was measured on these same folds, so it is the baseline.
default_cv = lambda_sweep.set_index("lambda_1").loc[1e-6, "cv_MAE"]
print(f"default over the outer folds: MAE {default_cv:.2f} EUR")

__Step 36:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor predicts every car at the training mean.
results_dir = "results"
cars_board = Scoreboard("cars4you resale", "MAE (EUR)", context=result_context(
    "../../data/interim/cars4you.csv", regression_recipe_log,
    y_cars_train.index.tolist(), y_cars_test.index.tolist(),
    outer_folds=outer_folds(cars_folds, cars_train, y_cars_train)),
    floor=round(mean_absolute_error(y_cars_test, DummyRegressor(strategy="mean").fit(
        cars_train, y_cars_train).predict(cars_test)), 4))
for week, model, cv, test in results_before(7, "regression", results_dir,
                                             metric=cars_board.metric,
                                             context=cars_board.context):
    cars_board.record(week, model, cv, test)

__Step 37:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names the winning configuration.
settings = cars_search.best_params_
chosen_label = (f"BayesianRidge(lambda_1={settings['estimator__lambda_1']:g},"
                f" alpha_1={settings['estimator__alpha_1']:g},"
                f" max_iter={settings['estimator__max_iter']})")
cars_board.record(7, chosen_label,
                  round(-cars_nested["test_mae"].mean(), 4), round(tuned_mae, 4))
cars_board.save(
    results_dir, 7, "regression", baseline_cv=round(default_cv, 4),
    baseline_test=round(default_mae, 4),
)

# The fitted winner is saved beside its result, so a later week can reload it if
# it ends the course as the best model. Loading needs preprocessing.py on the path.
joblib.dump(chosen_bayesian, f"{results_dir}/week_07_regression_model.joblib")

print(cars_board.table())

<div class="alert alert-block alert-success">

**`BayesianRidge` joins the board at 3,016.00 EUR over the ten outer folds and
3,069.30 held out.** Both errors are larger than those of the Lasso on log(price),
2,546.51 and 2,652.89, and far below the floor's 6,843.46. The row and the fitted
winner are saved in `results/`.

The floor's 6,843.46 is also the test MAE of the `lambda_1` sweep's collapsed
model: with its coefficients at zero, that model predicted the training mean for
every car, which is what the floor does.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **At its defaults, `BayesianRidge` returns a range that holds what it
   claims:** its nominal 95 % intervals held 96.2 % of the validation and test prices.
2. **A zero-centred Gaussian prior is a ridge penalty:** `Ridge` with the constant
   `lambda_ / alpha_`, 0.1068, reproduced the coefficients to 6.97e-10.
3. **Far from the training cars the interval widens, and the prediction can become
   impossible:** ten spans before the oldest `year`, one car's standard deviation grew
   from 4,909 to 13,094 EUR while its prediction fell to -351,642 EUR.
4. **Inside a fold the width ranked the errors but barely varied:** none of 1,000
   shuffles matched its correlation of +0.1872, and the widest interval was at most
   1.152 times the narrowest.
5. **The two hyperpriors fail in different columns:** at `lambda_1=1e-2` every
   coefficient went to zero and the validation MAE rose to 7,231.10; at
   `alpha_1=1e7` the MAE moved by about three euros while the coverage fell to
   0.0292.
6. **Nested cross-validation put the grid at 3,016.00 EUR, 18.97 below
   the default's 3,034.97 on these folds**, and the grid's winner, chosen on MAE, has intervals
   that hold 52.8 % of the validation prices.

Exit ticket: name one thing `BayesianRidge` returned here that
`LinearRegression` did not. Then name one thing it did not give that you might have
expected.

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Bishop, C.M. (2006). *Pattern Recognition and Machine Learning*. Springer, § 3.3-3.5 (Bayesian linear regression, the evidence approximation, and the predictive distribution).
- MacKay, D.J.C. (1992). Bayesian interpolation. *Neural Computation*, 4(3), 415-447.
- Tipping, M.E. (2001). Sparse Bayesian learning and the relevance vector machine. *JMLR*, 1, 211-244.
- Hoerl, A.E. & Kennard, R.W. (1970). Ridge regression: biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55-67. (The frequentist half of the equivalence shown above.)
- scikit-learn User Guide, [Bayesian regression](https://scikit-learn.org/stable/modules/linear_model.html#bayesian-regression) · [`BayesianRidge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.BayesianRidge.html).